In [19]:
from dataclasses import dataclass
import csv

In [9]:
class Block:
    def __init__(self, id, view):
        self.id = id
        self.view = view

    def __str__(self):
        return f"{{'id': '{self.id}', 'view': {self.view}}}"

@dataclass
class Vote:
    block_id: str
    def __hash__(self):
        return hash(self.block_id)

In [14]:
def read_input_csv(filename):
    inputs = []
    with open(filename, 'r', newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            t = row['type'].strip().lower()
            if t == 'block':
                bid = row['block_id'].strip()
                view = int(row['view'])
                inputs.append(Block(bid, view))
            elif t == 'vote':
                bid = row['vote_id'].strip()
                inputs.append(Vote(bid))
    return inputs

In [15]:
class ChainBuilder:
    def __init__(self):
        self.chain = []
        self.votes = set()
        self.pending = {}

    def process_vote(self, vote):
        self.votes.add(vote.block_id)
        if vote.block_id in self.pending:
            self.try_add_block(self.pending[vote.block_id])

    def process_block(self, block):
        self.pending[block.id] = block
        self.try_add_block(block)

    def try_add_block(self, block):
        if block.id in self.votes and block.view == len(self.chain):
            self.chain.append(block)
            del self.pending[block.id]
            self.check_pending_after_add()

    def check_pending_after_add(self):
        added = True
        while added:
            added = False
            for bid in list(self.pending.keys()):
                blk = self.pending[bid]
                if blk.id in self.votes and blk.view == len(self.chain):
                    self.chain.append(blk)
                    del self.pending[bid]
                    added = True
                    break

    def run(self, inputs):
        for item in inputs:
            if isinstance(item, Vote):
                self.process_vote(item)
            else:
                self.process_block(item)
        return self.chain

In [16]:
def print_chain(chain):
    print("Отриманий ланцюжок:")
    for block in chain:
        print(block)

In [18]:
inputs = read_input_csv('lab2.csv')

builder = ChainBuilder()
chain = builder.run(inputs)

print_chain(chain)

Отриманий ланцюжок:
{'id': '0x156', 'view': 0}
{'id': '0x123', 'view': 1}
{'id': '0x123', 'view': 2}
